In [ ]:
!pip install streamlit yfinance pandas numpy matplotlib pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 22.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [streamlit]


In [ ]:
from google.colab import files

files.upload()

Saving logo.png to logo.png


{'logo.png': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\xbf\x00\x00\x00\xab\x08\x06\x00\x00\x00\xf6#\x92K\x00\x00\x00\x01sRGB\x00\xae\xce\x1c\xe9\x00\x00\x00\x04gAMA\x00\x00\xb1\x8f\x0b\xfca\x05\x00\x00\x00\tpHYs\x00\x00\x0e\xc3\x00\x00\x0e\xc3\x01\xc7o\xa8d\x00\x008`IDATx^\xed\x9d\x7fl\x1b\xe7y\xc7\x9f\x9a\xb4L\xd9\xb2R\xd90\x9dX\x86\xc3\xa4q\xcc8N\xe4,MY\xafm~hY\xdat\xc4\xbc5P\nl\x98\xd6\x16\xdd\x90\x82]\x97lY\xa0\r\r&\x0c\x85\x16,\x9dV\x0c\xb0\xe6\x00J\xb6\x14\x1a\x10\xc4c\x96t\xcc\x8c\xc1\x91#\x0f\x05m\xa7\x8ed\xcfq\xa8\xccIj\xaaQ\xa2\xc8\r\x95\x98\x0ee\xd2d\xde\xfdQ>\xb7\xe7\x9e{\x8f\xbc\xe3\x0f\x89\xa4\x9e\x0f\xf0\x05\x8f\xef\xbdw\xf7\xfe\xb8{\x9e{\xdf{\xef\xbdO\x01\x80\x02A\x10\x04AXA\xac\xe2\x01\x82 \x08\x82\xd0\xea\x88\xf3\x13\x84:\xe3\xf5z\xc1\xeb\xf5\xf2`A\x10\x96\x91OI\xb7\xa7 4>\xe8<\xf3\xf9<_%\x08B\x05\x88\xf3\x13\x84%D\x9c\x98 4\x06\xe2\xfc\x04A\x10\x84\x15\x878?A\xa8\x11\xe5\x9e\xeb\xe5\xf3ym\x1c\xda\n\xd4\xad\x07i)\nB\xcd\x11\xe7\'\x08.\xe1\x0ej9\x1c\x13M\xc3r\x

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3EBn2tJ4j07DqhyCxRgIInfpUtO_CDNhGD9jqVw5TrUnTEs1")

In [ ]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Configuración inicial ---
st.set_page_config(page_title="VISION FINANCIERA", layout="wide")

# --- Fondo principal y estilos ---
page_bg_img = """
<style>
[data-testid="stAppViewContainer"] {
    background-image: url("https://images.unsplash.com/photo-1639322537228-f710d846310a?auto=format&fit=crop&w=1920&q=80");
    background-size: cover;
    background-repeat: no-repeat;
    background-attachment: fixed;
    padding-top: 0px !important;
}
.logo-container {
    text-align: center;
    margin: 0px;
    padding: 0px;
}
.logo-container img {
    width: 800px;
    display: block;
    margin-left: auto;
    margin-right: auto;
    filter: drop-shadow(0px 0px 15px gold);
    animation: pulseGlow 2s infinite;
}
@keyframes pulseGlow {
    0% { filter: drop-shadow(0px 0px 10px gold); }
    50% { filter: drop-shadow(0px 0px 25px gold); }
    100% { filter: drop-shadow(0px 0px 10px gold); }
}
.block-container {
    padding-top: 0rem !important;
    margin-top: -15px !important;
}
</style>
"""
st.markdown(page_bg_img, unsafe_allow_html=True)

# --- Espacio superior ---
st.markdown("<br><br>", unsafe_allow_html=True)

# --- Logo centrado ---
col1, col2, col3 = st.columns([4,1,4])

with col2:
    st.image("logo.png", width=170)

# --- Título ---
st.markdown("""
<h1 style="
text-align:center;
color:gold;
font-size:60px;
font-weight:bold;
margin-top:-5px;
text-shadow:0px 0px 15px gold;">
VISIÓN FINANCIERA
</h1>
""", unsafe_allow_html=True)

# --- Estado de navegación ---
if "menu" not in st.session_state:
    st.session_state.menu = "Dashboard"

# --- Menú ---
menu_options = ["Dashboard", "Estrategias", "Backtesting", "Elemento Cripto", "Disclaimer IA"]
cols = st.columns(len(menu_options))
for i, option in enumerate(menu_options):
    if cols[i].button(option):
        st.session_state.menu = option

# --- Configuración inicial ---
symbols = ["BTC-USD","ETH-USD","SOL-USD","ADA-USD","LTC-USD"]
crypto = st.selectbox("Criptomoneda:", symbols)
start_date = st.date_input("Fecha inicio", pd.to_datetime("2023-01-01"))
end_date = st.date_input("Fecha fin", pd.to_datetime("2024-01-01"))
investment = st.number_input("Monto hipotético ($)", value=1000)

# --- Descargar datos ---
data = yf.download(crypto, start=start_date, end=end_date)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = [' '.join(col).strip() for col in data.columns.values]
if data.empty:
    st.error(f"No se pudieron descargar datos para {crypto}.")
    st.stop()

precio_col = next((col for col in data.columns if "Adj Close" in col or "Close" in col), None)
if precio_col:
    data = data[[precio_col]].rename(columns={precio_col:"Precio"})
else:
    st.error("No se encontró columna de precios.")
    st.stop()

data["Rendimiento"] = data["Precio"].pct_change()

# --- Secciones ---
menu = st.session_state.menu

if menu == "Dashboard":
    st.title("📊 Dashboard de Análisis")
    st.line_chart(data["Precio"])

    freq = st.radio("Frecuencia de rendimientos:", ["Diarios", "Semanales", "Mensuales"])
    if freq == "Diarios":
        returns = data["Precio"].pct_change()
    elif freq == "Semanales":
        returns = data["Precio"].resample("W").ffill().pct_change()
    else:
        returns = data["Precio"].resample("M").ffill().pct_change()

    st.line_chart(returns)

    cum_return = (1 + data["Rendimiento"]).cumprod() - 1
    st.line_chart(cum_return)
    st.write(f"Ganancia/Pérdida sobre ${investment}: {investment*cum_return.iloc[-1]:.2f}")

    vol = data["Rendimiento"].std() * np.sqrt(252)
    cum_return_series = (1+data["Rendimiento"]).cumprod()
    drawdown = (cum_return_series/cum_return_series.cummax() - 1)
    st.metric("Volatilidad anualizada", f"{vol:.2%}")
    st.metric("Máximo drawdown", f"{drawdown.min():.2%}")
    st.metric("Días negativos", f"{(data['Rendimiento'] < 0).mean():.2%}")

elif menu == "Estrategias":
    st.title("💡 Estrategias de Trading")
    short_win = st.slider("Ventana corta (días)", 5, 30, 10)
    long_win = st.slider("Ventana larga (días)", 50, 200, 100)
    data["MA_Corta"] = data["Precio"].rolling(short_win).mean()
    data["MA_Larga"] = data["Precio"].rolling(long_win).mean()
    st.line_chart(data[["Precio","MA_Corta","MA_Larga"]])
    data["Signal"] = np.where(data["MA_Corta"] > data["MA_Larga"], 1, 0)
    data["Strategy_Return"] = data["Signal"].shift(1) * data["Rendimiento"]

elif menu == "Backtesting":
    st.title("📈 Backtesting")
    if "Strategy_Return" not in data.columns:
        short_win, long_win = 10, 50
        data["MA_Corta"] = data["Precio"].rolling(short_win).mean()
        data["MA_Larga"] = data["Precio"].rolling(long_win).mean()
        data["Signal"] = np.where(data["MA_Corta"] > data["MA_Larga"], 1, 0)
        data["Strategy_Return"] = data["Signal"].shift(1) * data["Rendimiento"]
    cum_strategy = (1+data["Strategy_Return"]).cumprod()-1
    cum_buyhold = (1+data["Rendimiento"]).cumprod()-1
    st.line_chart(pd.DataFrame({"Estrategia":cum_strategy,"Buy&Hold":cum_buyhold}))

elif menu == "Elemento Cripto":
    st.title("🔍 Elemento Propio del Mercado Cripto")

    # --- Consolidadas vs Especulativas ---
    st.subheader("🏆 Consolidadas vs Especulativas")
    consolidadas = ["BTC-USD", "ETH-USD"]
    especulativas = ["SOL-USD", "ADA-USD", "DOGE-USD"]

    # Descargar y aplanar
    consol_df = yf.download(consolidadas, start=start_date, end=end_date)
    espec_df = yf.download(especulativas, start=start_date, end=end_date)

    if isinstance(consol_df.columns, pd.MultiIndex):
        consol_df.columns = [' '.join(col).strip() for col in consol_df.columns.values]
    if isinstance(espec_df.columns, pd.MultiIndex):
        espec_df.columns = [' '.join(col).strip() for col in espec_df.columns.values]

    # Seleccionar columnas de precios
    consol_prices = consol_df[[c for c in consol_df.columns if "Adj Close" in c or "Close" in c]].pct_change().mean()
    espec_prices = espec_df[[c for c in espec_df.columns if "Adj Close" in c or "Close" in c]].pct_change().mean()

    st.write(f"Rendimiento promedio consolidadas (BTC, ETH): {consol_prices.mean():.2%}")
    st.write(f"Rendimiento promedio especulativas (SOL, ADA, DOGE): {espec_prices.mean():.2%}")
    st.write("👉 Las especulativas muestran mayor volatilidad que las consolidadas.")

    # --- Caídas extremas y recuperación ---
    st.subheader("📉 Caídas extremas y recuperación")
    extreme_loss = (data["Rendimiento"] < -0.05).mean()
    st.write(f"{extreme_loss:.2%} de los días tuvieron pérdidas mayores al 5%")
    cum_return = (1 + data["Rendimiento"]).cumprod()
    drawdown = (cum_return / cum_return.cummax() - 1).min()
    st.write(f"Máximo drawdown: {drawdown:.2%}")

    # --- Pérdidas superiores a umbral configurable ---
    st.subheader("⚠️ Pérdidas superiores al umbral")
    umbral = st.slider("Selecciona umbral de pérdida (%)", 1, 20, 5) / 100
    loss_days = (data["Rendimiento"] < -umbral).mean()
    st.write(f"{loss_days:.2%} de los días tuvieron pérdidas mayores al {umbral*100:.0f}%")

    # --- Mercado 24/7 ---
    st.subheader("🕒 Mercado 24/7 (fines de semana)")
    data["DiaSemana"] = data.index.dayofweek
    weekend_returns = data[data["DiaSemana"] >= 5]["Rendimiento"].mean()
    st.write(f"Rendimiento promedio en fines de semana: {weekend_returns:.2%}")

    # --- Correlación entre criptomonedas ---
    st.subheader("🔗 Correlación entre criptomonedas")
    crypto1 = st.selectbox("Cripto 1", symbols, index=0)
    crypto2 = st.selectbox("Cripto 2", symbols, index=1)

    df1 = yf.download(crypto1, start=start_date, end=end_date)
    df2 = yf.download(crypto2, start=start_date, end=end_date)

    if isinstance(df1.columns, pd.MultiIndex):
        df1.columns = [' '.join(col).strip() for col in df1.columns.values]
    if isinstance(df2.columns, pd.MultiIndex):
        df2.columns = [' '.join(col).strip() for col in df2.columns.values]

    col1 = next((c for c in df1.columns if "Adj Close" in c or "Close" in c), None)
    col2 = next((c for c in df2.columns if "Adj Close" in c or "Close" in c), None)

    if col1 and col2:
        correlation = df1[col1].pct_change().corr(df2[col2].pct_change())
        st.write(f"Correlación {crypto1} vs {crypto2}: {correlation:.2f}")

elif menu == "Disclaimer IA":

    st.title("🤖 Transparencia en el Uso de IA")

    st.markdown("""
    ### Herramientas utilizadas

    - Microsoft Copilot
    - ChatGPT

    ### Uso realizado

    Las herramientas de inteligencia artificial fueron utilizadas como apoyo para:

    - Resolución de dudas de programación.
    - Optimización y corrección de errores.
    - Sugerencias de visualización.
    - Apoyo en la construcción de algunas funciones del dashboard.
    - Revisión de código y documentación.

    ### Trabajo realizado por el equipo

    El equipo fue responsable de:

    - Seleccionar las criptomonedas analizadas.
    - Definir la tesis de inversión.
    - Diseñar la estrategia de trading.
    - Escoger los indicadores financieros.
    - Interpretar los resultados.
    - Validar los cálculos realizados.
    - Ajustar la interfaz y experiencia de usuario.
    - Elaborar las conclusiones y recomendaciones.

    ### Declaración

    La inteligencia artificial fue utilizada únicamente como una herramienta de apoyo técnico y consulta.

    Todas las decisiones analíticas, financieras y estratégicas fueron tomadas y verificadas por los integrantes del equipo VISIÓN FINANCIERA.
    """)

    st.success(
        "✅ El equipo validó los resultados, métricas y conclusiones presentadas en esta aplicación."
    )

Overwriting app.py


In [ ]:
public_url = ngrok.connect(8501)
print("Tu app está disponible en:", public_url)
!streamlit run app.py --server.port 8501 --server.headless true


Tu app está disponible en: NgrokTunnel: "https://winner-remark-alfalfa.ngrok-free.dev" -> "http://localhost:8501"


2026-11-10 00:57:25.996 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.7.117:8501

/content/app.py:87: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(crypto, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/content/app.py:87: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(crypto, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/content/app.py:87: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(crypto, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/content/app.py:87: FutureWarning: YF.download() has changed argument auto_adjust defa